# Big Data

## What is the Pile?
The Pile is an English text corpus that was created by EleutherAI for training large-scale language models. It includes a diverse range of datasets, spanning scientific articles, GitHub code repositories, and filtered web text. The training corpus is available in 14 GB chunks, and you can also download several of the individual components. Let’s start by taking a look at the PubMed Abstracts dataset, which is a corpus of abstracts from 15 million biomedical publications on PubMed. The dataset is in JSON Lines format and is compressed using the zstandard library, so first we need to install that:

```!pip install zstandard```

In [ ]:
from datasets import load_dataset

pubmed_dataset = load_dataset(
    "casinca/PUBMED_title_abstracts_2019_baseline",
    split="train"
)

pubmed_dataset

## The magic of memory mapping

In [ ]:
import psutil

print(f"RAM used: {psutil.Process().memory_info().rss / (1024 * 1024):.2f} MB")

In [ ]:
print(f"Dataset size in bytes: {pubmed_dataset.dataset_size}")
size_gb = pubmed_dataset.dataset_size / (1024 ** 3)
print(f"Dataset size (cache file): {size_gb:.2f} GB")

In [ ]:
pubmed_dataset[0]

In [ ]:
import timeit

code_snippet = """batch_size=1000

for idx in range(0, len(pubmed_dataset), batch_size):
    _ = pubmed_dataset[idx:idx + batch_size]
"""

time = timeit.timeit(stmt=code_snippet, number=1, globals=globals())

print(
    f"Iterated ober {len(pubmed_dataset)} examples 9about {size_gb:.1f} GB) in "
    f"{time:.1f}s, i.e. {size_gb/time:.3f}GB/s"
)

## Streaming datasets

In [ ]:
pumped_dataset_stream = load_dataset("casinca/PUBMED_title_abstracts_2019_baseline", 
                                     split="train",
                                     streaming=True)

In [ ]:
next(iter(pumped_dataset_stream))

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tokenized_dataset = pumped_dataset_stream.map(lambda x: tokenizer(x["text"]))
next(iter(tokenized_dataset))

In [ ]:
shuffeld_dataset = pumped_dataset_stream.shuffle(buffer_size=10_000, seed=42)
next(iter(shuffeld_dataset))

In [ ]:
dataset_head = pumped_dataset_stream.take(5)
dataset_head

In [ ]:
list(dataset_head)

In [ ]:
train_dataset = shuffeld_dataset.skip(1000)

validation_dataset = shuffeld_dataset.take(1000)

In [ ]:
train_dataset[0]

In [ ]:
next(iter(train_dataset))

In [ ]:
from datasets import load_dataset

law_dataset_streamed = load_dataset(
    "timaeus/pile-freelaw",
    split="train",
    streaming=True,
)

In [ ]:
next(iter(law_dataset_streamed))

In [ ]:
from itertools import islice
from datasets import interleave_datasets

combined_dataset = interleave_datasets([pumped_dataset_stream, law_dataset_streamed])
list(islice(combined_dataset, 2))

In [ ]:
pumped_dataset_stream = pumped_dataset_stream.remove_columns(["meta"])
law_dataset_streamed = law_dataset_streamed.remove_columns(["meta"])

In [ ]:
from itertools import islice
from datasets import interleave_datasets

combined_dataset = interleave_datasets([pumped_dataset_stream, law_dataset_streamed])
list(islice(combined_dataset, 2))